Hyperparams selection

# Libraries

In [13]:
import openpyxl
import pandas as pd
from pathlib import Path
import numpy as np
import sys
from tqdm import tqdm
import optuna
import json
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [14]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [15]:
# hyperparams
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_hyperparams import *

In [16]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

In [17]:
RANDOM_STATE = 42

# Datatasets selection

In [60]:
path_data = Path('../data')
path_results = Path('../results/')
path_highlighted = path_results / 'metrics_modelling2_highlight.xlsx'
df_highlighted = pd.read_excel(path_highlighted)
wb = openpyxl.load_workbook(path_highlighted, data_only=True)
ws = wb.worksheets[0]

In [61]:
highlighted_rows = []
for row in range(1, ws.max_row):
    cell = ws.cell(column=1, row=row)
    bgColor = cell.fill.bgColor.index
    fgColor = cell.fill.fgColor.index
    if bgColor != '00000000':
        highlighted_rows.append(row)
highlighted_rows = np.array(highlighted_rows) - 2
df_filtered = df_highlighted.iloc[highlighted_rows]

Datasets for parameters selection.

In [62]:
df_filtered

,dataset,target,model,accuracy,f1,precision,recall,roc_auc
22,df_modelling_dimensionless,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.90,0.931818
23,df_modelling_dimensionless_pf,net_impact,catboostclassifier_net_impact,0.946667,0.900000,0.900000,0.90,0.931818
26,df_modelling_dimensionless,net_impact,catboostclassifier_smote_net_impact,0.933333,0.878049,0.857143,0.90,0.922727
27,df_modelling_no_multicollinearity,net_impact,kneighborsclassifier_net_impact_ordenc,0.933333,0.878049,0.857143,0.90,0.922727
34,df_modelling_no_multicollinearity,net_impact,xgbclassifier_smote_net_impact,0.933333,0.871795,0.894737,0.85,0.906818
35,df_modelling_no_multicollinearity_pf,net_impact,xgbclassifier_ohe_smote_net_impact,0.933333,0.871795,0.894737,0.85,0.906818
37,df_modelling_no_multicollinearity_pf,net_impact,kneighborsclassifier_net_impact_onehot,0.920000,0.857143,0.818182,0.90,0.913636
39,df_modelling_no_multicollinearity,net_impact,xgbclassifier_ohe_smote_net_impact,0.920000,0.850000,0.850000,0.85,0.897727
75,df_modelling_no_multicollinearity,net_impact,svc_smote_net_impact_onehot,0.893333,0.826087,0.730769,0.95,0.911364
77,df_modelling_no_multicollinearity_pf,net_impact,logisticregression_net_impact_ordenc,0.906667,0.820513,0.842105,0.80,0.872727


In [9]:
sorted(df_filtered['model'].unique())

['catboostclassifier_net_impact',
 'catboostclassifier_smote_net_impact',
 'catboostclassifier_smote_splashing',
 'catboostclassifier_splashing',
 'kneighborsclassifier_net_impact_onehot',
 'kneighborsclassifier_net_impact_ordenc',
 'kneighborsclassifier_smote_net_impact_ordenc',
 'kneighborsclassifier_smote_splashing_ordenc',
 'kneighborsclassifier_splashing_onehot',
 'kneighborsclassifier_splashing_ordenc',
 'logisticregression_net_impact_ordenc',
 'logisticregression_splashing_onehot',
 'svc_smote_net_impact_onehot',
 'svc_smote_net_impact_ordenc',
 'svc_smote_splashing_onehot',
 'svc_smote_splashing_ordenc']

# Parameters selection

In [10]:
metrics = []

In [11]:
def objective(
        trial, train, test, target, model_str, 
        random_state, dataset_filename, cat_features=['wettability']):
    params = get_params(trial, model_str, random_state, cat_features)
    model = get_model(model_str, params)
    preproc_pipeline = get_preproc_pipeline(model_str=model_str, random_state=random_state, 
                                            cat_features=cat_features, 
                                            num_features=dict_num_features[dataset_filename])
    pipeline = [('model', model)]
    if preproc_pipeline: pipeline.insert(0, preproc_pipeline[0])
    pipeline = Pipeline(steps=pipeline)
    pipeline.fit(train.drop(columns=[target]), train[target])
    preds = pipeline.predict(test.drop(columns=[target]))
    f1 = f1_score(test[target], preds)
    metrics.append({
            'accuracy': accuracy_score(test[target], preds),
            'f1': f1,
            'precision': precision_score(test[target], preds),
            'recall': recall_score(test[target], preds),
            'roc_auc': roc_auc_score(test[target], preds)})
    return f1

In [12]:
dict_results = {}
for i in (pbar:=tqdm(range(df_filtered.shape[0]))):
    dataset_filename = df_filtered.iloc[i]['dataset']
    if 'df_full' in dataset_filename: continue
    model_str = df_filtered.iloc[i]['model']
    pbar.set_description(f"Processing {model_str}")
    target = df_filtered.iloc[i]['target']
    train, test = get_train_test(
        dataset_filename=dataset_filename,
        target=target)
    study = optuna.create_study(direction="maximize")
    def obj(trial):
        return objective(
            trial, train, test, 
            target, model_str, dataset_filename=dataset_filename, 
            random_state=RANDOM_STATE, cat_features=['wettability'])
    study.optimize(obj, n_trials=10, show_progress_bar=True, timeout=30, n_jobs=-1)
    dict_results[model_str] = {
            'dataset': dataset_filename,
            'model': model_str,
            'target': target,
            'metrics': metrics[study.best_trial.number]}
    clear_output()
    metrics = []
with open("../results/optuna_results.json", "w") as outfile: 
    json.dump(dict_results, outfile)

Processing svc_smote_net_impact_ordenc: 100%|██████████| 22/22 [01:36<00:00,  4.38s/it]
